# Week 3 Session 5: Neural Networks and CNNs

This session is **visual-first**. We train a network in the very first activity, then peel back the layers (forward propagation, cost function, gradient descent, back-propagation) using pictures and small experiments rather than equations.

## Activities in this Session

| # | Activity | Topic | Style |
|---|----------|-------|-------|
| 1 | Dense NN baseline on CIFAR-10 | Build, train, observe | Worked + 1 TODO |
| 2 | Look inside the network | Forward prop, cost, gradient descent, back-prop | Worked + 2 TODOs |
| 3 | Add convolution -> CNN | Convolution, pooling, predict-your-own-image | Worked + 2 TODOs |

**Dataset:** CIFAR-10 (10 classes of 32x32 colour images, loaded from `keras.datasets`). To keep training fast in class, we use a 10,000-image subset.

> **Heads up:** the first time you run `cifar10.load_data()` Keras will download ~170 MB. Start the first code cell early.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from scipy.signal import convolve2d

# Reproducibility (matches the course convention)
np.random.seed(42)
tf.random.set_seed(42)

print('TensorFlow version:', tf.__version__)

---
# Activity 1 -- Dense NN Baseline on CIFAR-10

> ### Activity Card
> **Goal:** Build the simplest possible neural network and watch it learn.
> **Dataset:** CIFAR-10 (10,000 train / 2,000 test subset for speed).
> **Features (X):** 32x32x3 colour images (= 3,072 pixel values per image).
> **Target (y):** integer 0-9 -- one of `airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck`.
> **Task:** Load and view the data, train a 2-layer dense network, plot training curves, then change the activation function.

This activity is the teaching example -- code is filled in and explained.

In [ ]:
# Load CIFAR-10 and take a small subset so training fits inside class time
(X_train_full, y_train_full), (X_test_full, y_test_full) = cifar10.load_data()

# Shuffle once with a fixed seed, then slice
rng = np.random.default_rng(42)
train_idx = rng.permutation(len(X_train_full))[:10000]
test_idx  = rng.permutation(len(X_test_full))[:2000]

X_train = X_train_full[train_idx].astype('float32') / 255.0  # scale pixels to 0-1
X_test  = X_test_full[test_idx].astype('float32')  / 255.0
y_train = y_train_full[train_idx].flatten()
y_test  = y_test_full[test_idx].flatten()

class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print(f'X_train shape: {X_train.shape}')
print(f'X_test  shape: {X_test.shape}')
print(f'pixel range:   {X_train.min()} to {X_train.max()}')

In [ ]:
# Show 16 images in a 4x4 grid with their class labels
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for ax, img, label in zip(axes.flat, X_train[:16], y_train[:16]):
    ax.imshow(img)
    ax.set_title(class_names[label], fontsize=9)
    ax.axis('off')
plt.suptitle('CIFAR-10 -- 16 random training images')
plt.tight_layout()
plt.show()

### TODO 1.1 -- Plot 8 images of ONE class

Pick any class index (0-9) and show 8 training images that belong to it. Helps you see how varied "cat" or "truck" pictures actually are.

In [ ]:
# TODO: choose a class index from 0 to 9
chosen_class = 3   # 3 = 'cat' -- change this to any class you want

# TODO: filter X_train to images where y_train == chosen_class, take the first 8
# Hint: idx = np.where(y_train == chosen_class)[0][:8]


# TODO: plot the 8 images in a 2x4 grid using plt.subplots(2, 4, figsize=(10, 5))
# Hint: copy the loop pattern from the cell above


### What is a Neuron?
A neuron does three things: it takes some inputs, multiplies each by a weight, adds a bias, and squashes the result through an activation function.

```
output = activation( w1*x1 + w2*x2 + ... + wn*xn + b )
```

A **dense layer** is just many neurons stacked side by side, each one looking at all the same inputs but with its own set of weights.

For CIFAR-10 we will start with the simplest possible architecture:

```
input (32x32x3) -> Flatten -> Dense(128, relu) -> Dense(10, softmax) -> class probabilities
```

`relu` is `max(0, x)` -- a simple non-linearity. `softmax` converts the final 10 numbers into probabilities that sum to 1.

In [ ]:
# Build a tiny dense network -- 2 layers
model_dense = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Flatten(),                         # 32*32*3 = 3072 numbers
    layers.Dense(128, activation='relu'),     # hidden layer with 128 neurons
    layers.Dense(10,  activation='softmax'),  # output: probability per class
])

model_dense.compile(optimizer='adam',
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])

model_dense.summary()

In [ ]:
# Train for 5 epochs on the small CIFAR subset
history_dense = model_dense.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_test, y_test),
    verbose=2,
)

test_loss, test_acc = model_dense.evaluate(X_test, y_test, verbose=0)
print(f'\nDense NN test accuracy: {test_acc:.4f}')

In [ ]:
# Plot the training curves
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history_dense.history['loss'],     label='train')
axes[0].plot(history_dense.history['val_loss'], label='test')
axes[0].set_title('Loss per epoch')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss'); axes[0].legend()

axes[1].plot(history_dense.history['accuracy'],     label='train')
axes[1].plot(history_dense.history['val_accuracy'], label='test')
axes[1].set_title('Accuracy per epoch')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('accuracy'); axes[1].legend()

plt.tight_layout()
plt.show()

### TODO 1.2 -- Swap the activation function

Build the same network but with **`sigmoid`** instead of `relu` in the hidden layer, train it for 5 epochs, and compare the test accuracy. Sigmoid was the classic choice before ReLU took over -- you should see why.

In [ ]:
# TODO: build a copy of model_dense but use activation='sigmoid' in the hidden layer
# Hint: same Sequential() block, just change one word
model_sigmoid = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Flatten(),
    # TODO: Dense(128, activation='sigmoid'),
    # TODO: Dense(10, activation='softmax'),
])


# TODO: compile with the same optimizer, loss, metrics as model_dense


# TODO: fit for 5 epochs and store the history in history_sigmoid


# TODO: evaluate on (X_test, y_test) and print the test accuracy


---
# Activity 2 -- Look Inside the Network

> ### Activity Card
> **Goal:** Open the black box. See forward propagation, the cost function, and gradient descent **as pictures**, not as equations.
> **Task:** Run a forward pass by hand on one image, plot the cost surface, watch a tiny network roll downhill, and compare three learning rates.

### What is Forward Propagation?
Forward propagation is just **input -> layer 1 -> layer 2 -> output**. At each layer we compute `W @ x + b` then apply the activation. Let's do the first hidden layer **by hand** for one image so you can watch the shapes change.

In [ ]:
# Grab the weights and bias of the FIRST dense layer of model_dense
W1, b1 = model_dense.layers[1].get_weights()
print(f'W1 shape: {W1.shape}   (3072 inputs -> 128 neurons)')
print(f'b1 shape: {b1.shape}')

# Take ONE image and flatten it to 3072 numbers
one_image = X_train[0].reshape(-1)   # shape: (3072,)
print(f'\nflattened image shape: {one_image.shape}')

# Forward pass through layer 1: z = x @ W + b, then ReLU
z1 = one_image @ W1 + b1
a1 = np.maximum(0, z1)               # ReLU activation

print(f'layer-1 pre-activation  z1 shape: {z1.shape}')
print(f'layer-1 post-activation a1 shape: {a1.shape}')
print(f'first 5 activations: {a1[:5].round(3)}')

### TODO 2.1 -- Forward pass on a different image

Pick image index 7 (or any other), run the same forward pass through layer 1, and print the first 5 post-activation values. Notice that the **shape stays the same** even though the values are completely different -- that's the whole point of a layer.

In [ ]:
# TODO: pick a different image
img_idx = 7

# TODO: flatten it, do x @ W1 + b1, apply ReLU
# Hint: copy the 3 lines above and change `one_image`


# TODO: print the first 5 activations rounded to 3 decimals


### What is the Cost Function?
The **cost** (or **loss**) is a single number that measures how wrong the network's predictions are. Training tries to make this number small.

For binary cases the simplest cost is squared error: `(prediction - target)**2`. The plot below shows what that looks like for a single example where the true label is 1.

In [ ]:
# Cost function visual: squared error when the target is 1
predictions = np.linspace(0, 1, 100)
target = 1.0
loss   = (predictions - target) ** 2

plt.figure(figsize=(7, 4))
plt.plot(predictions, loss, color='steelblue', linewidth=2)
plt.scatter([0.2, 0.5, 0.9], [(0.2-1)**2, (0.5-1)**2, (0.9-1)**2],
            color=['red','orange','green'], s=80, zorder=5)
for p, name in zip([0.2, 0.5, 0.9], ['bad', 'meh', 'good']):
    plt.annotate(f'{name}\npred={p}', xy=(p, (p-1)**2),
                 xytext=(p-0.05, (p-1)**2 + 0.08), fontsize=9)
plt.title('Cost = (prediction - 1)^2  -- lower is better')
plt.xlabel('predicted probability of correct class')
plt.ylabel('cost')
plt.grid(True, alpha=0.3)
plt.show()

### What is Gradient Descent?
Gradient descent is the algorithm that **rolls the weights downhill** on the cost surface. At every step:

1. Compute how much the cost changes when each weight changes (the **gradient**).
2. Take a small step **against** that gradient.
3. Repeat.

The size of the step is the **learning rate**. The animation below shows a single weight rolling down a parabolic cost.

In [ ]:
# Tiny gradient-descent demo on cost = w**2 (minimum at w=0)
def cost(w):    return w ** 2
def grad(w):    return 2 * w

w   = 4.0          # start far from the minimum
lr  = 0.3          # learning rate
trajectory = [w]
for _ in range(8):
    w = w - lr * grad(w)
    trajectory.append(w)

ws = np.linspace(-5, 5, 100)
plt.figure(figsize=(7, 4))
plt.plot(ws, cost(ws), color='lightgray', linewidth=2)
for i, w_step in enumerate(trajectory):
    plt.scatter(w_step, cost(w_step), color='steelblue',
                s=120 - i*10, zorder=5)
    plt.annotate(f'step {i}', (w_step, cost(w_step)),
                 textcoords='offset points', xytext=(5, 5), fontsize=8)
plt.title('Gradient descent on cost = w^2  (lr = 0.3)')
plt.xlabel('weight w'); plt.ylabel('cost')
plt.grid(True, alpha=0.3)
plt.show()

### TODO 2.2 -- Try three learning rates

Train the dense network three times with learning rates **0.001**, **0.1**, and **10.0** for **3 epochs each**, then plot the three loss curves on the same axes. You should see:

- `0.001` -- too small, learns very slowly
- `0.1`   -- often the sweet spot
- `10.0`  -- way too big, loss explodes / NaN

Hint: build the model inside the loop so each run starts fresh.

In [ ]:
from tensorflow.keras.optimizers import Adam

# TODO: for each learning rate, build a fresh dense model, compile it with
#       Adam(learning_rate=lr), fit for 3 epochs, and store history.history['loss'].

learning_rates = [0.001, 0.1, 10.0]
loss_curves = {}

for lr in learning_rates:
    # TODO: build a small Sequential like model_dense
    # TODO: compile with optimizer=Adam(learning_rate=lr)
    # TODO: fit for 3 epochs (set verbose=0 to keep the output tidy)
    # TODO: loss_curves[lr] = history.history['loss']
    pass

# TODO: plot the three curves on the same axes with plt.plot, label each one
# Hint: for lr, curve in loss_curves.items(): plt.plot(curve, label=f'lr={lr}')


### A Peek at Back-Propagation
Back-propagation is how the network figures out the gradient for every single weight, no matter how many layers there are. The maths uses the chain rule -- but **Keras and TensorFlow do all of that for you**. The cell below shows it happening: we ask `tf.GradientTape` to compute one gradient on a 2-weight toy.

In [ ]:
# A 2-weight toy: predict y = w1*x + w2 and compute the gradient
x_toy = tf.constant(2.0)
y_toy = tf.constant(7.0)
w1 = tf.Variable(1.0)
w2 = tf.Variable(0.0)

with tf.GradientTape() as tape:
    prediction = w1 * x_toy + w2
    loss_toy   = (prediction - y_toy) ** 2

grads = tape.gradient(loss_toy, [w1, w2])
print(f'prediction:    {prediction.numpy():.2f}  (target was {y_toy.numpy():.2f})')
print(f'loss:          {loss_toy.numpy():.2f}')
print(f'd_loss / d_w1: {grads[0].numpy():.2f}')
print(f'd_loss / d_w2: {grads[1].numpy():.2f}')
print('\nKeras runs this for every weight in the network, every batch. That is back-propagation.')

### Reflection (Activity 2)

1. In **TODO 2.2**, what happened to the loss curve when you used `lr = 10.0`? Why?
2. Forward propagation gave us **predictions**. Back-propagation gave us **gradients**. What does gradient descent then do with those gradients?
3. Roughly how many weights does our dense model have? (Look at `model_dense.summary()`.) Imagine computing each gradient by hand.

---
# Activity 3 -- Add Convolution -> CNN

> ### Activity Card
> **Goal:** Replace the dense baseline with a Convolutional Neural Network and watch the test accuracy jump.
> **Why convolution?** A dense layer treats every pixel as independent. But pixels have **neighbours** -- a convolution slides a small kernel across the image and learns local patterns (edges, textures, shapes).
> **Task:** See a kernel in action, build a CNN, compare it to the dense baseline, predict on your own test image.

In [ ]:
# A 3x3 edge-detection kernel applied by hand to one image (greyscale)
edge_kernel = np.array([[-1, -1, -1],
                        [-1,  8, -1],
                        [-1, -1, -1]])

# Convert one CIFAR image to greyscale by averaging the RGB channels
gray = X_train[0].mean(axis=2)

# Slide the kernel over the image
edges = convolve2d(gray, edge_kernel, mode='same', boundary='symm')

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(gray,  cmap='gray');  axes[0].set_title('original (grey)');  axes[0].axis('off')
axes[1].imshow(edges, cmap='gray');  axes[1].set_title('after edge kernel'); axes[1].axis('off')
plt.tight_layout()
plt.show()

### TODO 3.1 -- Try a blur and a sharpen kernel

Use the same `convolve2d` call but with these two kernels and plot the result side by side with the original.

In [ ]:
# Two famous kernels -- already filled in for you
blur_kernel = np.ones((3, 3)) / 9.0     # average of the 9 neighbours

sharpen_kernel = np.array([[ 0, -1,  0],
                           [-1,  5, -1],
                           [ 0, -1,  0]])

# TODO: apply blur_kernel to `gray` using convolve2d(gray, blur_kernel, mode='same', boundary='symm')


# TODO: apply sharpen_kernel the same way


# TODO: plot original / blurred / sharpened in a 1x3 grid, all in cmap='gray'


### What is a CNN?
A Convolutional Neural Network stacks two ideas:

| Layer | What it does |
|-------|-------------|
| `Conv2D` | Learns its own kernels (the network decides what edges/textures matter). |
| `MaxPool2D` | Downsamples by keeping the max value in each 2x2 region -- shrinks the image and keeps the strongest signal. |

We repeat **Conv -> Pool** a couple of times, then flatten and feed a small dense head.

```
Input -> Conv -> Pool -> Conv -> Pool -> Flatten -> Dense -> output
```

In [ ]:
# Build a small CNN -- 2 conv blocks + a tiny dense head
model_cnn = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, kernel_size=3, activation='relu', padding='same'),
    layers.MaxPooling2D(pool_size=2),
    layers.Conv2D(64, kernel_size=3, activation='relu', padding='same'),
    layers.MaxPooling2D(pool_size=2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax'),
])

model_cnn.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
model_cnn.summary()

In [ ]:
# Train the CNN for 5 epochs (slower than the dense net but still classroom-friendly)
history_cnn = model_cnn.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_test, y_test),
    verbose=2,
)

cnn_test_loss, cnn_test_acc = model_cnn.evaluate(X_test, y_test, verbose=0)
print(f'\nCNN test accuracy:        {cnn_test_acc:.4f}')
print(f'Dense baseline accuracy:  {test_acc:.4f}')

# Save weights so Session 6 can reload this exact CNN
model_cnn.save('week3_cnn.keras')
print('\nCNN saved to week3_cnn.keras (used in Session 6).')

In [ ]:
# Bar chart -- dense baseline vs CNN
plt.figure(figsize=(6, 4))
plt.bar(['Dense NN', 'CNN'],
        [test_acc, cnn_test_acc],
        color=['lightcoral', 'mediumseagreen'])
plt.ylabel('test accuracy')
plt.title('CIFAR-10 -- Dense baseline vs CNN')
plt.ylim(0, 1)
for i, v in enumerate([test_acc, cnn_test_acc]):
    plt.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)
plt.show()

### TODO 3.2 -- Predict on your own test image

Pick any image from `X_test`, ask the CNN to predict it, and print the top-3 most likely classes with their probabilities. Then look at the actual image -- did the CNN get it right? Did the second guess make sense?

In [ ]:
# TODO: pick a test-image index between 0 and 1999
my_idx = 42

# TODO: get the prediction probabilities
# Hint: model_cnn.predict(X_test[my_idx:my_idx+1]) returns shape (1, 10)


# TODO: find the top-3 classes
# Hint: top3 = probs.argsort()[-3:][::-1]


# TODO: show the image and print the top-3 class names + probabilities
# Hint: plt.imshow(X_test[my_idx]); print('actual:', class_names[y_test[my_idx]])


### Discussion Questions (Activity 3)

1. By how many percentage points did the CNN beat the dense NN? Why is that gap there?
2. Look at one image where the CNN was wrong. Was its second guess at least sensible (e.g. confused a cat with a dog)?
3. The CNN has many fewer parameters than the dense model in the early layers. How can a smaller model do better?

---

## Wrap-up

You built two models, watched forward propagation by hand, plotted a cost function, ran gradient descent on a parabola, peeked at back-propagation through `GradientTape`, and saw convolution turn pixels into features.

In **Session 6** we keep the same CNN and play with its hyperparameters, then meet a different kind of model -- the Vision Transformer.